In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from scipy.fftpack import fft
from sklearn.preprocessing import MinMaxScaler

# ============================
# Step 1: Data Preprocessing
# ============================

# Simulating data (Replace this with real dataset loading)
# Generate dummy time-series data
np.random.seed(42)
time_series_data = np.random.rand(1000, 256)  # 1000 samples, 256 time steps each
labels = np.random.randint(0, 11, 1000)  # 3 classes for fault types

# Segment signals (if needed, based on your real dataset)
# Normalization
scaler = MinMaxScaler()
time_series_data = scaler.fit_transform(time_series_data)

# Feature Extraction
def extract_features(segment):
    """Extract statistical and FFT-based features from a segment."""
    mean = np.mean(segment)
    var = np.var(segment)
    fft_coeff = np.abs(fft(segment))[:len(segment) // 2]
    return np.concatenate(([mean, var], fft_coeff))

# Extract features for all samples
features = np.array([extract_features(sample) for sample in time_series_data])

# Train-test split
X_train_signal, X_test_signal, X_train_features, X_test_features, y_train, y_test = train_test_split(
    time_series_data, features, labels, test_size=0.2, random_state=42
)

# ============================
# Step 2: Dataset Preparation
# ============================

class FaultDataset(Dataset):
    def __init__(self, signals, features, labels):
        self.signals = torch.tensor(signals, dtype=torch.float32)
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.signals[idx], self.features[idx], self.labels[idx]

# Create Dataloaders
train_dataset = FaultDataset(X_train_signal, X_train_features, y_train)
test_dataset = FaultDataset(X_test_signal, X_test_features, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ============================
# Step 3: Model Definition
# ============================

class HybridModel(nn.Module):
    def __init__(self, signal_dim, feature_dim, hidden_dim, num_classes):
        super(HybridModel, self).__init__()
        # Signal Encoder
        self.signal_encoder = nn.Sequential(
            nn.Linear(signal_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Feature Extractor
        self.feature_extractor = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, signals, features):
        encoded_signals = self.signal_encoder(signals)
        extracted_features = self.feature_extractor(features)
        combined = torch.cat((encoded_signals, extracted_features), dim=1)
        return self.classifier(combined)

# Model Initialization
model = HybridModel(signal_dim=256, feature_dim=X_train_features.shape[1], hidden_dim=128, num_classes=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ============================
# Step 4: Pretraining with SSL
# ============================

# SimCLR-style self-supervised loss
def contrastive_loss(anchor, positive, temperature=0.5):
    cosine_similarity = nn.CosineSimilarity(dim=-1)
    sim = cosine_similarity(anchor.unsqueeze(1), positive.unsqueeze(0)) / temperature
    labels = torch.arange(anchor.size(0)).to(anchor.device)
    return nn.CrossEntropyLoss()(sim, labels)

# Self-supervised pretraining
for epoch in range(10):  # Pretraining for 10 epochs
    model.train()
    total_loss = 0
    for signals, features, _ in train_loader:
        # Generate augmented versions of signals
        augmented_signals = signals + 0.05 * torch.randn_like(signals)  # Add noise
        augmented_features = features + 0.05 * torch.randn_like(features)

        # Forward pass
        anchor = model.signal_encoder(signals)
        positive = model.signal_encoder(augmented_signals)
        loss = contrastive_loss(anchor, positive)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Pretraining Epoch [{epoch+1}/10], Loss: {total_loss/len(train_loader):.4f}")

# ============================
# Step 5: Fine-tuning
# ============================

# Fine-tuning on labeled data
for epoch in range(20):  # Fine-tuning for 20 epochs
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for signals, features, labels in train_loader:
        # Forward pass
        outputs = model(signals, features)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    print(f"Fine-tuning Epoch [{epoch+1}/20], Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100.*correct/total:.2f}%")

# ============================
# Step 6: Evaluation
# ============================

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for signals, features, labels in test_loader:
        outputs = model(signals, features)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

print(f"Test Accuracy: {100.*correct/total:.2f}%")
